In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df=pd.read_csv("../data/Dataset.csv")

print(df.head())
print(df.tail())

  show_id     type                             title         director  \
0      s1    Movie              Dick Johnson Is Dead  Kirsten Johnson   
1      s3  TV Show                         Ganglands  Julien Leclercq   
2      s6  TV Show                     Midnight Mass    Mike Flanagan   
3     s14    Movie  Confessions of an Invisible Girl    Bruno Garotti   
4      s8    Movie                           Sankofa     Haile Gerima   

         country date_added  release_year rating  duration  \
0  United States  9/25/2021          2020  PG-13    90 min   
1         France  9/24/2021          2021  TV-MA  1 Season   
2  United States  9/24/2021          2021  TV-MA  1 Season   
3         Brazil  9/22/2021          2021  TV-PG    91 min   
4  United States  9/24/2021          1993  TV-MA   125 min   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dr

In [3]:
print(df['rating'].value_counts())

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [4]:
df.dropna(subset=['rating']).copy()

X=df.drop(columns=['rating','show_id','title'])
y=df['rating']

print("X Shape:",X.shape)
print("y shape:",y.shape)
print("\nMissing values in target:", y.isnull().sum())
print("\nRating categories:", y.nunique())

print(X.dtypes)

print(X.isnull().sum())

X Shape: (8790, 7)
y shape: (8790,)

Missing values in target: 0

Rating categories: 14
type              str
director          str
country           str
date_added        str
release_year    int64
duration          str
listed_in         str
dtype: object
type            0
director        0
country         0
date_added      0
release_year    0
duration        0
listed_in       0
dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


categorical_feature=['type','director','country','date_added','duration','listed_in']
numerical_feature=['release_year']
preprocessor=ColumnTransformer(transformers=[("cat",OneHotEncoder(handle_unknown='ignore'),categorical_feature),("num","passthrough",numerical_feature)])
print ("Preprocessor created successfully")
print(preprocessor.transformers)



X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
X_trained_transform=preprocessor.fit_transform(X_train)
print("Original Shape:",X.shape)
print("After tranformation",X_trained_transform.shape)

Preprocessor created successfully
[('cat', OneHotEncoder(handle_unknown='ignore'), ['type', 'director', 'country', 'date_added', 'duration', 'listed_in']), ('num', 'passthrough', ['release_year'])]
Training data: (7032, 7)
Testing data: (1758, 7)
Original Shape: (8790, 7)
After tranformation (7032, 6159)


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report

dt_model=Pipeline([("preprocessor",preprocessor),
                   ("classifier",DecisionTreeClassifier(random_state=42))])
dt_model.fit(X_train,y_train)

y_pred=dt_model.predict(X_test)
print(y_pred)
dt_accuracy=accuracy_score(y_test,y_pred)

print("Decision Tree Accuracy:",dt_accuracy)
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

['TV-14' 'TV-14' 'TV-Y' ... 'TV-MA' 'R' 'TV-MA']
Decision Tree Accuracy: 0.49658703071672355
              precision    recall  f1-score   support

           G       0.00      0.00      0.00        12
       NC-17       0.00      0.00      0.00         0
          NR       0.08      0.05      0.06        21
          PG       0.47      0.35      0.40        54
       PG-13       0.34      0.32      0.33        93
           R       0.37      0.52      0.43       155
       TV-14       0.51      0.51      0.51       444
        TV-G       0.22      0.09      0.13        44
       TV-MA       0.59      0.64      0.62       628
       TV-PG       0.37      0.25      0.30       181
        TV-Y       0.55      0.59      0.57        66
       TV-Y7       0.46      0.44      0.45        59
          UR       0.00      0.00      0.00         1

    accuracy                           0.50      1758
   macro avg       0.30      0.29      0.29      1758
weighted avg       0.48      0.50      0.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report


rf_model=Pipeline([("preprocessor",preprocessor),('classifier',RandomForestClassifier(n_estimators=200, random_state=42,class_weight='balanced',n_jobs=-1))])
rf_model.fit(X_train,y_train)
y_pred=rf_model.predict(X_test)
rf_accuracy=accuracy_score(y_test,y_pred)
print("Random Forest Classifier",rf_accuracy)
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Random Forest Classifier 0.5130830489192264

Classification Report:
              precision    recall  f1-score   support

           G       0.50      0.08      0.14        12
          NR       0.11      0.05      0.07        21
          PG       0.50      0.46      0.48        54
       PG-13       0.38      0.48      0.43        93
           R       0.38      0.71      0.50       155
       TV-14       0.56      0.56      0.56       444
        TV-G       0.24      0.20      0.22        44
       TV-MA       0.67      0.54      0.60       628
       TV-PG       0.32      0.29      0.30       181
        TV-Y       0.54      0.59      0.57        66
       TV-Y7       0.43      0.51      0.47        59
          UR       0.00      0.00      0.00         1

    accuracy                           0.51      1758
   macro avg       0.39      0.37      0.36      1758
weighted avg       0.53      0.51      0.51      1758



hyperparameter tunning

In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_tunning_pipeline=Pipeline([("preprocessor",preprocessor),
                              ('classifier',RandomForestClassifier(n_estimators=200, random_state=42,class_weight='balanced',n_jobs=-1))])
params={
    'classifier__n_estimators':[100,200],
    'classifier__max_depth':[None,20],
    'classifier__min_samples_split':[2,5]
}

grid_search=GridSearchCV(estimator=rf_tunning_pipeline,
                         param_grid=params,
                         cv=3,
                         scoring='accuracy',
                         n_jobs=-1,
                         verbose=1)
grid_search.fit(X_train,y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

C:\Users\Designer\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best Parameters: {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best Cross-Validation Accuracy: 0.5007110352673493


In [11]:

from sklearn.metrics import accuracy_score, classification_report

best_rf_model = grid_search.best_estimator_
y_pred_tuned_rf = best_rf_model.predict(X_test)
tuned_rf_accuracy = accuracy_score(y_test, y_pred_tuned_rf)
print("Tuned Random Forest Test Accuracy:", tuned_rf_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_tuned_rf,
    zero_division=0
))


Tuned Random Forest Test Accuracy: 0.5130830489192264

Classification Report:
              precision    recall  f1-score   support

           G       0.50      0.08      0.14        12
          NR       0.11      0.05      0.07        21
          PG       0.50      0.46      0.48        54
       PG-13       0.38      0.48      0.43        93
           R       0.38      0.71      0.50       155
       TV-14       0.56      0.56      0.56       444
        TV-G       0.24      0.20      0.22        44
       TV-MA       0.67      0.54      0.60       628
       TV-PG       0.32      0.29      0.30       181
        TV-Y       0.54      0.59      0.57        66
       TV-Y7       0.43      0.51      0.47        59
          UR       0.00      0.00      0.00         1

    accuracy                           0.51      1758
   macro avg       0.39      0.37      0.36      1758
weighted avg       0.53      0.51      0.51      1758



In [12]:
print("Decision Tree Accuracy:", dt_accuracy)
print("Original Random Forest Accuracy:", rf_accuracy)
print("Tuned Random Forest Accuracy:", tuned_rf_accuracy)

Decision Tree Accuracy: 0.49658703071672355
Original Random Forest Accuracy: 0.5130830489192264
Tuned Random Forest Accuracy: 0.5130830489192264


 Conclusion — Task 3: Netflix Audience Rating Classification

In this task, I developed a multiclass classification model to predict Netflix audience rating categories using content attributes. I analyzed the rating distribution, prepared the training and testing datasets, and trained Decision Tree and Random Forest classifiers.
The Decision Tree achieved 49.66% accuracy, while the original Random Forest achieved 51.31% accuracy. I then applied GridSearchCV with 3-fold cross-validation to tune the Random Forest hyperparameters. The tuned model achieved the same test accuracy of 51.31%.
Random Forest performed better than Decision Tree on the test dataset. However, the results indicate that further improvements may be possible through feature engineering, handling class imbalance, and additional model optimization.